# Gauge-induced suboptimality

This companion figure reuses the broadcast-reconstruction experiment and isolates the excess error of the RBLA index prefix over the dense rank-$r$ optimum:

$$G_r = E_r^{\mathrm{RBLA}} - E_r^{\mathrm{DenseSVD}} \approx E_r^{\mathrm{RBLA}} - E_r^{\mathrm{SP+}}.$$

The curve is the mean over gauge-rotation seeds and the shaded region is one standard deviation. No new synthetic matrices are generated.

In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import font_manager

figure_dir = Path.cwd().resolve()
if figure_dir.name != 'gauge_induced_suboptimality':
    figure_dir = figure_dir / 'rbla+' / 'figures' / 'gauge_induced_suboptimality'
figure_dir.mkdir(parents=True, exist_ok=True)
source_csv = figure_dir.parent / 'broadcast_reconstruction' / 'broadcast_reconstruction_error.csv'
output_csv = figure_dir / 'gauge_induced_suboptimality.csv'
output_pdf = figure_dir / 'gauge_induced_suboptimality.pdf'

source = pd.read_csv(source_csv)
source['checkpoint'] = source['checkpoint'].fillna('N/A')
index_columns = [
    'dataset', 'model', 'layer_or_aggregate', 'seed', 'round',
    'R', 'r', 'normalized_rank', 'experiment_type', 'checkpoint',
    'd_out', 'd_in', 'spectrum',
]
paired = (
    source.pivot_table(
        index=index_columns, columns='method', values='relative_error', aggfunc='first'
    )
    .reset_index()
    .rename_axis(columns=None)
)
required_methods = ['RBLA index prefix', 'SP+ (compact QR)', 'Dense truncated SVD']
missing = [method for method in required_methods if method not in paired.columns]
if missing:
    raise ValueError(f'Missing methods in source CSV: {missing}')

paired = paired.rename(columns={
    'RBLA index prefix': 'rbla_relative_error',
    'SP+ (compact QR)': 'sp_plus_relative_error',
    'Dense truncated SVD': 'dense_relative_error',
})
paired['excess_error_gap'] = paired['rbla_relative_error'] - paired['dense_relative_error']
paired['sp_plus_excess_error'] = paired['sp_plus_relative_error'] - paired['dense_relative_error']

summary = (
    paired.groupby(['R', 'r', 'normalized_rank'], as_index=False)['excess_error_gap']
    .agg(mean_excess_error='mean', std_excess_error='std',
         min_excess_error='min', max_excess_error='max', seed_count='count')
    .sort_values('r')
)
paired = paired.merge(summary, on=['R', 'r', 'normalized_rank'], how='left')
paired.to_csv(output_csv, index=False, float_format='%.16e')

tolerance = 1e-11
max_sp_dense_gap = float(np.abs(paired['sp_plus_excess_error']).max())
min_excess_gap = float(paired['excess_error_gap'].min())
full_rank_gap = float(np.abs(paired.loc[paired['r'].eq(paired['R']), 'excess_error_gap']).max())
checks = {
    'sp_plus_matches_dense': max_sp_dense_gap <= tolerance,
    'rbla_not_better_than_dense': min_excess_gap >= -tolerance,
    'full_rank_gap_is_zero': full_rank_gap <= tolerance,
}
if not all(checks.values()):
    raise AssertionError(checks)

available_fonts = {font.name for font in font_manager.fontManager.ttflist}
paper_font = 'Times New Roman' if 'Times New Roman' in available_fonts else 'DejaVu Serif'
rc = {
    'font.family': paper_font,
    'font.size': 16,
    'axes.labelsize': 22,
    'xtick.labelsize': 17,
    'ytick.labelsize': 17,
    'legend.fontsize': 16,
    'axes.linewidth': 1.1,
    'xtick.major.width': 1.0,
    'ytick.major.width': 1.0,
    'xtick.major.size': 4.0,
    'ytick.major.size': 4.0,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
}
with mpl.rc_context(rc):
    fig, ax = plt.subplots(figsize=(5.0, 5.0), dpi=600)
    x = summary['normalized_rank'].to_numpy(float)
    mean = summary['mean_excess_error'].to_numpy(float)
    std = summary['std_excess_error'].fillna(0.0).to_numpy(float)
    color = '#C44E52'
    ax.fill_between(
        x, np.maximum(0.0, mean - std), mean + std,
        color=color, alpha=0.15, linewidth=0, zorder=1,
    )
    ax.plot(
        x, mean, color=color, marker='o', linewidth=2.2, markersize=6.5,
        label='RBLA excess', zorder=3,
    )
    ax.axhline(0.0, color='black', linestyle='--', linewidth=2.0,
               label='Optimal baseline', zorder=2)
    ax.set_xlabel(r'Normalized client rank $r/R$')
    ax.set_ylabel('Excess error over optimum')
    ax.set_xlim(float(x.min()) - 0.015, 1.015)
    upper_limit = float((mean + std).max()) * 1.08
    ax.set_ylim(-0.02 * upper_limit, upper_limit)
    ax.set_xticks(x)
    ax.set_xticklabels([f'{value:.3g}' for value in x])
    ax.grid(axis='both', color='0.75', linestyle=':', linewidth=0.7, alpha=0.30)
    ax.set_axisbelow(True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(direction='out', width=1.0, length=4)
    ax.legend(loc='upper right', frameon=False, handlelength=2.2, borderaxespad=0.3)
    fig.tight_layout(pad=0.25)
    fig.savefig(output_pdf, format='pdf', bbox_inches='tight', pad_inches=0.03)
    plt.show()

print(f'Source: {source_csv}')
print(f'Seeds per rank: {int(summary.seed_count.min())}')
print(f'Maximum mean excess error: {summary.mean_excess_error.max():.6f}')
print(f'Maximum SP+-dense difference: {max_sp_dense_gap:.3e}')
print(f'Numerical checks: {checks}')
summary